# 05 - SFCN Colab Bootstrap

This notebook bootstraps the first brain-age reproduction milestone for this repository: a usable `SFCN` baseline on the cleaned `SIMON` T1 manifest.

This version is **no-Drive by default**. It expects one downloadable archive containing:
- `simon/manifest.csv`
- `simon/images/`
- `simon/metadata/`
- `weights/run_20190719_00_epoch_best_mae.p`

It does five things:
1. installs the repo and Colab runtime dependencies
2. downloads and extracts one `SIMON` bundle into `/content/brain_assets`
3. rewrites the manifest paths for Colab-local execution
4. writes `config/local/brain_age_runtime.json`
5. runs standalone `SynthStrip` validation and `SFCN` smoke tests


In [ ]:
from pathlib import Path

DATA_SOURCE_MODE = "url_bundle"
SIMON_BUNDLE_URL = ""
SIMON_BUNDLE_SHA256 = ""
LOCAL_ASSET_ROOT = Path("/content/brain_assets")
REPO_URL = "https://github.com/kondratevakate/faceage-to-brainage.git"
REPO_DIR = Path("/content/faceage-to-brainage")
RUNTIME_DEVICE = "cuda"

SMOKE_LIMIT_SIMON = 3
PILOT_LIMIT_SIMON = 10

print("DATA_SOURCE_MODE:", DATA_SOURCE_MODE)
print("SIMON_BUNDLE_URL:", SIMON_BUNDLE_URL or "<set me>")
print("LOCAL_ASSET_ROOT:", LOCAL_ASSET_ROOT)
print("REPO_URL:", REPO_URL)
print("RUNTIME_DEVICE:", RUNTIME_DEVICE)


## Expected bundle layout

The downloadable archive must contain exactly this structure:

```text
bundle_root/
  simon/
    manifest.csv
    images/
      sub-..._T1w.nii.gz
      ...
    metadata/
      README
      dataset_description.json
      SIMON_pheno (4).csv
      t1_sidecars/
        sub-..._T1w.json
        ...
  weights/
    run_20190719_00_epoch_best_mae.p
```

Only direct public `HTTP/HTTPS` bundle URLs are supported in this notebook.


In [ ]:
if DATA_SOURCE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
elif DATA_SOURCE_MODE == "url_bundle":
    print("Skipping Google Drive mount because DATA_SOURCE_MODE='url_bundle'.")
else:
    raise ValueError(f"Unsupported DATA_SOURCE_MODE: {DATA_SOURCE_MODE}")


In [ ]:
import os
import shutil
import stat
import subprocess

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["python", "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run(["pip", "install", "-r", str(REPO_DIR / "requirements.txt")], check=True)
subprocess.run([
    "pip",
    "install",
    "surfa @ https://github.com/freesurfer/surfa/archive/ca4e26317d3556776268ede7de20752743bf49a2.zip",
], check=True)
subprocess.run([
    "git",
    "clone",
    "https://github.com/ha-ha-ha-han/UKBiobank_deep_pretrain.git",
    str(REPO_DIR / "vendor" / "SFCN"),
], check=True)

synthstrip_root = REPO_DIR / "vendor" / "SynthStrip"
(synthstrip_root / "models").mkdir(parents=True, exist_ok=True)
(synthstrip_root / "bin").mkdir(parents=True, exist_ok=True)
(REPO_DIR / "config" / "local").mkdir(parents=True, exist_ok=True)

subprocess.run([
    "wget",
    "-O",
    str(synthstrip_root / "mri_synthstrip"),
    "https://raw.githubusercontent.com/freesurfer/freesurfer/dev/mri_synthstrip/mri_synthstrip",
], check=True)
subprocess.run([
    "wget",
    "-O",
    str(synthstrip_root / "models" / "synthstrip.1.pt"),
    "https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/requirements/synthstrip.1.pt",
], check=True)

wrapper_path = synthstrip_root / "bin" / "mri_synthstrip"
wrapper_path.write_text(
    "#!/usr/bin/env bash\n"
    "set -euo pipefail\n"
    f"export FREESURFER_HOME={synthstrip_root}\n"
    f"exec python {synthstrip_root / 'mri_synthstrip'} \"$@\"\n",
    encoding="utf-8",
)
os.chmod(synthstrip_root / "mri_synthstrip", os.stat(synthstrip_root / "mri_synthstrip").st_mode | stat.S_IEXEC)
os.chmod(wrapper_path, os.stat(wrapper_path).st_mode | stat.S_IEXEC)

print("Repo installed at", REPO_DIR)
print("Standalone SynthStrip installed at", synthstrip_root)


In [ ]:
synthstrip_bin = str(REPO_DIR / "vendor" / "SynthStrip" / "bin")
if synthstrip_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = synthstrip_bin + os.pathsep + os.environ.get("PATH", "")
os.environ["FREESURFER_HOME"] = str(REPO_DIR / "vendor" / "SynthStrip")

print("PATH head:", os.environ["PATH"].split(os.pathsep)[:3])
print("FREESURFER_HOME:", os.environ["FREESURFER_HOME"])


## Download and extract the `SIMON` bundle

The notebook downloads one bundle to `/content/brain_assets` using `wget`, optionally verifies SHA-256, then extracts it.


In [ ]:
import hashlib
import tarfile
import zipfile

if DATA_SOURCE_MODE != "url_bundle":
    raise RuntimeError("This notebook cell expects DATA_SOURCE_MODE='url_bundle'.")
if not SIMON_BUNDLE_URL.strip():
    raise ValueError("Set SIMON_BUNDLE_URL to a public direct HTTP/HTTPS bundle URL before running this cell.")

if LOCAL_ASSET_ROOT.exists():
    shutil.rmtree(LOCAL_ASSET_ROOT)
LOCAL_ASSET_ROOT.mkdir(parents=True, exist_ok=True)

bundle_download = LOCAL_ASSET_ROOT / "simon_bundle.download"
subprocess.run(["wget", "-O", str(bundle_download), SIMON_BUNDLE_URL], check=True)

if SIMON_BUNDLE_SHA256.strip():
    digest = hashlib.sha256(bundle_download.read_bytes()).hexdigest()
    if digest.lower() != SIMON_BUNDLE_SHA256.strip().lower():
        raise ValueError(f"SHA-256 mismatch: expected {SIMON_BUNDLE_SHA256}, got {digest}")
    print("SHA-256 verified:", digest)

if tarfile.is_tarfile(bundle_download):
    with tarfile.open(bundle_download) as tar:
        tar.extractall(LOCAL_ASSET_ROOT)
elif zipfile.is_zipfile(bundle_download):
    with zipfile.ZipFile(bundle_download) as zf:
        zf.extractall(LOCAL_ASSET_ROOT)
else:
    raise RuntimeError(f"Bundle is neither a tar archive nor a zip archive: {bundle_download}")

simon_candidates = [p.parent for p in LOCAL_ASSET_ROOT.rglob("manifest.csv") if p.parent.name == "simon"]
weight_candidates = list(LOCAL_ASSET_ROOT.rglob("run_20190719_00_epoch_best_mae.p"))
if len(simon_candidates) != 1:
    raise RuntimeError(f"Expected exactly one extracted simon/manifest.csv, found {len(simon_candidates)} candidates: {simon_candidates}")
if len(weight_candidates) != 1:
    raise RuntimeError(f"Expected exactly one extracted SFCN weight, found {len(weight_candidates)} candidates: {weight_candidates}")

SIMON_ROOT = simon_candidates[0]
EXTRACTED_SFCN_WEIGHT = weight_candidates[0]

print("SIMON_ROOT:", SIMON_ROOT)
print("EXTRACTED_SFCN_WEIGHT:", EXTRACTED_SFCN_WEIGHT)
for path in sorted(LOCAL_ASSET_ROOT.iterdir()):
    print(path)


## Rewrite manifest paths for Colab-local execution

The exported manifest may still contain Windows paths. This cell rewrites `t1_path` to the extracted local image folder and saves `manifest_colab.csv`.


In [ ]:
import pandas as pd

manifest_src = SIMON_ROOT / "manifest.csv"
manifest_dst = SIMON_ROOT / "manifest_colab.csv"
images_dir = SIMON_ROOT / "images"
sidecars_dir = SIMON_ROOT / "metadata" / "t1_sidecars"

manifest = pd.read_csv(manifest_src)
manifest["t1_path"] = manifest["t1_filename"].apply(lambda name: str(images_dir / name))

if "json_filename" in manifest.columns and "json_path" in manifest.columns:
    manifest["json_path"] = manifest["json_filename"].apply(lambda name: str(sidecars_dir / name))

missing_inputs = [p for p in manifest["t1_path"] if not Path(p).exists()]
if missing_inputs:
    raise FileNotFoundError(f"Missing extracted SIMON inputs referenced by manifest_colab.csv: {missing_inputs[:5]}")

manifest.to_csv(manifest_dst, index=False)
print("Wrote", manifest_dst)
print("Rows:", len(manifest))


In [ ]:
weight_dest = REPO_DIR / "vendor" / "SFCN" / "brain_age" / "run_20190719_00_epoch_best_mae.p"
weight_dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(EXTRACTED_SFCN_WEIGHT, weight_dest)
print("Copied weight to", weight_dest)


In [ ]:
import json

cfg = {
    "runtime": {
        "device": RUNTIME_DEVICE
    },
    "sfcn": {
        "model_dir": str(REPO_DIR / "vendor" / "SFCN"),
        "weight_path": str(REPO_DIR / "vendor" / "SFCN" / "brain_age" / "run_20190719_00_epoch_best_mae.p"),
        "skullstrip_command": "mri_synthstrip",
        "skip_skullstrip": False,
        "keep_skullstripped": False,
        "age_bins": {
            "start": 42.0,
            "step": 1.0,
            "count": 40
        }
    },
    "datasets": {
        "simon": {
            "manifest_csv": str(SIMON_ROOT / "manifest_colab.csv"),
            "input_path_column": "t1_path",
            "chron_age_column": "age",
            "subject_id_column": "subject_id",
            "session_id_column": "session_id",
            "scan_id_column": "",
            "run_column": "run",
            "acquisition_label_column": "acquisition_label",
            "preproc_dir": str(SIMON_ROOT / "sfcn_preproc"),
            "output_csv": str(SIMON_ROOT / "sfcn_predictions.csv")
        }
    }
}

config_path = REPO_DIR / "config" / "local" / "brain_age_runtime.json"
config_path.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
print("Wrote", config_path)


## Environment validation

This section checks bundle ingress, tool availability, and `CUDA` before any inference is attempted.


In [ ]:
import torch

checks = {
    "repo_exists": REPO_DIR.exists(),
    "manifest_colab_exists": (SIMON_ROOT / "manifest_colab.csv").exists(),
    "simon_images_exists": (SIMON_ROOT / "images").exists(),
    "simon_metadata_exists": (SIMON_ROOT / "metadata").exists(),
    "sfcn_repo_exists": (REPO_DIR / "vendor" / "SFCN").exists(),
    "sfcn_weight_exists": (REPO_DIR / "vendor" / "SFCN" / "brain_age" / "run_20190719_00_epoch_best_mae.p").exists(),
    "synthstrip_script_exists": (REPO_DIR / "vendor" / "SynthStrip" / "mri_synthstrip").exists(),
    "synthstrip_model_exists": (REPO_DIR / "vendor" / "SynthStrip" / "models" / "synthstrip.1.pt").exists(),
    "mri_synthstrip": shutil.which("mri_synthstrip"),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
}
checks


In [ ]:
command_path = shutil.which("mri_synthstrip")
if not command_path:
    raise RuntimeError("mri_synthstrip is not on PATH")

print("command -v mri_synthstrip ->", command_path)
subprocess.run(["mri_synthstrip", "--help"], check=True)


## Preprocessing validation on one `SIMON` scan

Run the standalone `SynthStrip` wrapper once before the `SFCN` smoke test.


In [ ]:
import nibabel as nib

simon_manifest = pd.read_csv(SIMON_ROOT / "manifest_colab.csv")
sample_input = Path(simon_manifest.iloc[0]["t1_path"])
sample_output = SIMON_ROOT / "sfcn_preproc" / "_synthstrip_validation.nii.gz"
sample_output.parent.mkdir(parents=True, exist_ok=True)

subprocess.run([
    "mri_synthstrip",
    "-i", str(sample_input),
    "-o", str(sample_output),
], check=True)

if not sample_output.exists():
    raise FileNotFoundError(f"SynthStrip output was not created: {sample_output}")

img = nib.load(str(sample_output))
data = img.get_fdata(dtype="float32")
print("SynthStrip validation output:", sample_output)
print("shape:", data.shape)
print("nonzero_voxels:", int((data != 0).sum()))


## `SIMON` smoke test

Run `3` rows first. This is the first acceptance gate for the no-Drive baseline.


In [ ]:
subprocess.run(
    ["python", "scripts/batch_sfcn.py", "--dataset", "simon", "--limit", str(SMOKE_LIMIT_SIMON)],
    check=True,
    cwd=str(REPO_DIR),
)


In [ ]:
simon_pred = pd.read_csv(SIMON_ROOT / "sfcn_predictions.csv")
print("SIMON smoke rows:", len(simon_pred))
print(simon_pred[["subject_id", "session_id", "run", "chron_age", "predicted_age", "brain_age_gap", "status", "error"]].head())


## Optional: pilot batch

Uncomment and run after the smoke test succeeds.


In [ ]:
# subprocess.run(
#     ["python", "scripts/batch_sfcn.py", "--dataset", "simon", "--limit", str(PILOT_LIMIT_SIMON)],
#     check=True,
#     cwd=str(REPO_DIR),
# )


## Optional: full batch

Run only after the pilot output looks sane.


In [ ]:
# subprocess.run(
#     ["python", "scripts/batch_sfcn.py", "--dataset", "simon"],
#     check=True,
#     cwd=str(REPO_DIR),
# )
